# 04-Warehouse-Agent-Database

**Purpose:** Build synthetic warehouse inventory data and load it into PostgreSQL for the warehouse agent. This notebook prepares the `warehouses.inventory` table used by the warehouse agent to answer availability and fulfillment queries.

**Concepts:** Qdrant product catalog extraction, synthetic data generation, PostgreSQL batch insert, warehouse schema.

**Course reference:** Week 5 — Warehouse Agent with database-backed inventory.

# Import

**Why:** Load dependencies for Qdrant (product catalog), PostgreSQL (inventory DB), and data generation. The `utils` path setup lets this notebook run from project root or from `notebooks/week5`.

In [ ]:
import sys
from pathlib import Path

# Ensure notebooks/week5 is on path so 'utils' package can be imported.
# Run from project root or notebooks/week5; both work.
_cwd = Path.cwd()
if not (_cwd / "utils").is_dir():
    _week5 = _cwd / "notebooks" / "week5"
    if _week5.exists():
        sys.path.insert(0, str(_week5))

# Pydantic for structured data; Qdrant for product catalog; psycopg2 for Postgres.
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document
from langsmith import traceable, get_current_run_tree
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage, convert_to_messages, convert_to_openai_messages
from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import openai
import random
import ast
import inspect
import instructor
import json
from utils.utils import get_tool_descriptions, format_ai_message
from langgraph.checkpoint.postgres import PostgresSaver
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, FusionQuery
import numpy as np


# Fictional Warehouses

**Why:** Define warehouse metadata (id, location, name) for synthetic inventory. These match the `warehouses.inventory` schema. Six warehouses across Germany and France simulate a regional distribution network.

In [ ]:
# Each warehouse has warehouse_id (e.g. DE-BER-01), location, and name.
# Used by generate_inventory_data to assign stock per warehouse.
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LY0-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

# Simulate Stock Availability for Each Warehouse

**Goal:** For each warehouse, assign synthetic stock for products from the Amazon Qdrant catalog. Step 1: fetch all product IDs from Qdrant.

## Retrieve All Item IDs from Amazon Qdrant Collection

**How:** Query Qdrant with a dummy vector to fetch points without semantic ranking. We only need `parent_asin` (product ID) for inventory generation.

In [ ]:
# Qdrant runs locally (Docker) or via docker-compose. Port 6333 is default.
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
# 1536 = text-embedding-3-small dimension. Zero vector returns points by internal order (no semantic ranking).
dummy_vector = np.zeros(1536).tolist()

In [ ]:
# Query returns up to 1000 points. We only need parent_asin (product ID); no vectors needed.
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [ ]:
payload

In [ ]:
payload.points

In [ ]:
# Extract product IDs from Qdrant points. These become product_id in warehouses.inventory.
parent_asin_list = [point.payload["parent_asin"] for point in payload.points]

In [ ]:
parent_asin_list

In [ ]:
len(parent_asin_list)

# Generate Synthetic Availability for All Items in Qdrant

**Why:** Simulate real-world distribution—not every product is in every warehouse. `availability_rate` (default 0.75) controls how often a product appears per warehouse. `total_quantity` is random 1–100 to vary stock levels.

In [ ]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    """
    Generate synthetic inventory records for warehouse × product combinations.

    Args:
        warehouses: List of dicts with warehouse_id, warehouse_location, warehouse_name.
        product_ids: List of product IDs (e.g. parent_asin from Qdrant).
        availability_rate: Probability (0–1) that a product is stocked in a warehouse. Default 0.75.

    Returns:
        List of dicts matching warehouses.inventory columns (minus id, available_quantity, updated_at).
    """
    inventory_records = []

    for warehouse in warehouses:
        for product_id in product_ids:
            # availability_rate controls distribution realism: not every product in every warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)

                # Skip zero-quantity rows; schema allows 0 but we want meaningful data
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })

    return inventory_records

In [ ]:
# Generate records for all warehouse × product combinations (75% availability).
inventory_data = generate_inventory_data(warehouses, parent_asin_list, availability_rate=0.75)

In [ ]:
inventory_data

In [ ]:
len(inventory_data)

# Write Synthetic Data into Postgres

**Why:** Load inventory into `warehouses.inventory` so the warehouse agent can query availability. Uses `execute_batch` for efficient bulk insert. Run `scripts/sql/warehouse_management.sql` first to create schema and table.

In [ ]:
def insert_inventory_to_db(inventory_records):
    """
    Bulk insert inventory records into warehouses.inventory.

    Uses execute_batch for performance (page_size=100). Connects to tools_database
    on port 5433 (same DB as shopping cart). available_quantity is computed by
    the table (GENERATED column); we only insert total_quantity and reserved_quantity.
    """
    try:
        # tools_database: same DB as shopping cart; warehouse schema is separate
        conn = psycopg2.connect(
            host="localhost",
            port=5433,
            database="tools_database",
            user="langgraph_user",
            password="langgraph_password"
        )
        conn.autocommit = True  # Each statement commits immediately; rollback on error

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """

            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)

            # Commit the transaction
            conn.commit()

            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")

            # Close cursor and connection
            cursor.close()
            conn.close()

    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if 'cursor' in locals() and cursor:
            cursor.close()
        if 'conn' in locals() and conn:
            conn.close()

In [ ]:
# Load synthetic inventory into warehouses.inventory. Run warehouse_management.sql first.
insert_inventory_to_db(inventory_data)